# Pakistan 2025 Monsoon Flood — Complete Analysis Notebook
---

 Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")
import plotly
print("pandas :", pd.__version__)
print("plotly :", plotly.__version__)
print("Ready!")

##  Load & Auto-Clean Dataset


**Cleaning steps applied:**
1. **Date parsing** — converts `start_date` and `peak_date` from text strings to proper datetime objects so timeline charts work correctly
2. **Type enforcement** — forces all numeric columns (deaths, injured, displaced etc.) to be integers, preventing errors in calculations
3. **Missing value fill** — fills any remaining NaN values so no chart breaks due to missing data

**Key variable created:** `df` — the main DataFrame used by every chart in this notebook.


In [ ]:
# Load dataset
df = pd.read_csv("/home/saif/Desktop/flood/pakistan_flood_2025_CLEAN.csv")

# Fix dates
df["start_date"] = pd.to_datetime(df["start_date"])
df["peak_date"]  = pd.to_datetime(df["peak_date"])

# Ensure numeric columns are correct type
int_cols = ["deaths","injured","displaced","houses_destroyed","houses_partially_damaged",
            "bridges_damaged","cropland_ha_damaged","livestock_lost","schools_damaged",
            "health_facilities_damaged","relief_camps","deaths_male","deaths_female",
            "deaths_children","injured_male","injured_female","injured_children",
            "rescue_operations","people_rescued","flood_duration_days"]

for c in int_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

df["roads_km_damaged"] = pd.to_numeric(df["roads_km_damaged"], errors="coerce").fillna(0).round(2)
df["damage_index"]     = pd.to_numeric(df["damage_index"],     errors="coerce").fillna(0).round(3)
df["mortality_rate"]   = pd.to_numeric(df["mortality_rate"],   errors="coerce").fillna(0).round(2)

# Fill any remaining nulls
df["notes"] = df["notes"].fillna("No additional notes")
df["relief_camps"] = df.groupby("province")["relief_camps"].transform(
    lambda x: x.fillna(x.median())).astype(int)

# Display summary
print(f"✓ Dataset loaded: {len(df)} districts | {len(df.columns)} columns")
print(f"✓ Missing values: {df.isnull().sum().sum()} (should be 0)")
print(f'✓ Provinces: {df["province"].nunique()}')
print(f'✓ Date range: {df["start_date"].min().date()} to {df["peak_date"].max().date()}')
print(f'✓ Total deaths: {df["deaths"].sum():,}')
print(f'✓ Total displaced: {df["displaced"].sum():,}')

df.head(3)

### Quick data check — single district lookup

This one-liner checks a specific value in the dataset. Used to verify the data loaded correctly.

In [ ]:
df.loc[df['district'] == 'Swat', 'houses_destroyed'].values[0]

##  Data Overview (Part 1): Statistical Summary


**How to read the output:**
- `count` — how many districts have a non-zero value (should be 50)
- `mean` — average value per district
- `max` — the worst single district (e.g. Buner = 256 deaths)
- `std` — standard deviation — high std means some districts are extreme outliers

> **Research insight:** High standard deviation in `deaths` (relative to mean) confirms that flood impact is highly unequal — a few districts absorb most of the casualties.

In [ ]:
# Statistical summary
key_cols = ["deaths","injured","displaced","houses_destroyed","roads_km_damaged",
            "bridges_damaged","cropland_ha_damaged","relief_camps","rescue_operations"]
df[key_cols].describe().round(1)

### Missing Value Check




In [ ]:
# Missing value check
missing = df.isnull().sum()
if missing.sum() == 0:
    print("No missing values — dataset is complete!")
else:
    print("Missing values found:")
    print(missing[missing>0])

###  Province & Confidence Breakdown



In [ ]:
# Province breakdown
print("Districts per province:")
print(df.groupby("province")["district"].count().sort_values(ascending=False).to_string())
print()
print("Confidence level breakdown:")
print(df["confidence_level"].value_counts().to_string())

## Province Summary Table


**New variable created:** `ps` (province summary) — used as input for most charts below.

**Columns calculated:**
- Human impact: `Deaths`, `Injured`, `Displaced` + gender breakdowns
- Infrastructure: `Houses_Destroyed`, `Houses_Partial`, `Roads_km`, `Bridges`, `Schools`
- Response: `Relief_Camps`, `Rescue_Ops`, `People_Rescued`
- Agriculture: `Cropland_ha`, `Livestock`
- Analysis: `Avg_Damage_Index`


In [ ]:
ps = df.groupby("province").agg(
    Districts          = ("district",              "count"),
    Deaths             = ("deaths",                "sum"),
    Deaths_Male        = ("deaths_male",           "sum"),
    Deaths_Female      = ("deaths_female",         "sum"),
    Deaths_Children    = ("deaths_children",       "sum"),
    Injured            = ("injured",               "sum"),
    Displaced          = ("displaced",             "sum"),
    Houses_Destroyed   = ("houses_destroyed",      "sum"),
    Houses_Partial     = ("houses_partially_damaged","sum"),
    Roads_km           = ("roads_km_damaged",      "sum"),
    Bridges            = ("bridges_damaged",       "sum"),
    Cropland_ha        = ("cropland_ha_damaged",   "sum"),
    Livestock          = ("livestock_lost",        "sum"),
    Schools            = ("schools_damaged",       "sum"),
    Relief_Camps       = ("relief_camps",          "sum"),
    Rescue_Ops         = ("rescue_operations",     "sum"),
    People_Rescued     = ("people_rescued",        "sum"),
    Avg_Damage_Index   = ("damage_index",          "mean"),
).reset_index().sort_values("Deaths", ascending=False)
ps["Avg_Damage_Index"] = ps["Avg_Damage_Index"].round(3)
ps["Roads_km"] = ps["Roads_km"].round(1)
ps

##  Color Theme & Dark Dashboard Setup


**Color assignments (chosen for maximum contrast):**
- KPK = Red `#FF4B4B` — highest deaths, warm/danger color
- Punjab = Blue `#4FC3F7` — riverine floods, water color
- Sindh = Green `#69F0AE`
- Balochistan = Yellow `#FFD740`
- Azad Kashmir = Purple `#CE93D8`
- Gilgit-Baltistan = Cyan `#80DEEA` — glacier color



In [ ]:
PROV_COLORS = {
    "Khyber Pakhtunkhwa": "#FF4B4B",
    "Punjab":             "#4FC3F7",
    "Sindh":              "#69F0AE",
    "Balochistan":        "#FFD740",
    "Azad Kashmir":       "#CE93D8",
    "Gilgit-Baltistan":   "#80DEEA",
}
BG   = "#0D1117"
CARD = "#161B22"
GRID = "#21262D"
TEXT = "#E6EDF3"
MUTED= "#8B949E"
AXIS = dict(gridcolor=GRID, zerolinecolor=GRID,
            tickfont=dict(color=MUTED, size=9),
            title_font=dict(color=MUTED, size=10),
            linecolor=GRID)
def base_layout(title, h=450):
    return dict(
        title=dict(text=title, font=dict(size=15,color=TEXT,family="Arial Black"), x=0.5),
        paper_bgcolor=BG, plot_bgcolor=CARD, font=dict(color=TEXT),
        height=h, margin=dict(t=80,b=70,l=60,r=40),
        legend=dict(bgcolor=CARD, font=dict(color=TEXT, size=10)),
    )
print("Color theme ready!")

## Deaths by Province


**Key findings visible in this chart:**
- KPK has the highest deaths — flash floods in steep mountain valleys are far more lethal per event than slow riverine floods
- Punjab has the second highest — due to the August Ravi + Chenab + Sutlej simultaneous flooding
- GB and AJK have fewer deaths despite GLOF events — smaller population exposed


In [ ]:
fig1 = go.Figure(go.Bar(
    x=ps["province"], y=ps["Deaths"],
    marker=dict(color=[PROV_COLORS[p] for p in ps["province"]],
                line=dict(color=BG, width=1)),
    text=ps["Deaths"], textposition="outside",
    textfont=dict(color=TEXT, size=11, family="Arial Black"),
    hovertemplate="<b>%{x}</b><br>Deaths: %{y:,}<extra></extra>",
))
fig1.update_layout(**base_layout("<b>Deaths by Province — 2025 Monsoon Floods</b>"))
fig1.update_xaxes(**AXIS, tickangle=-20)
fig1.update_yaxes(**AXIS, title="Deaths")
fig1.add_annotation(x=0.5, y=-0.18, xref="paper", yref="paper", showarrow=False,
    text="KPK = 534 deaths (flash floods deadlier than riverine) | Punjab = 339",
    font=dict(color=MUTED, size=10))
fig1.show()

# Fixed: Use single quotes for outer string or escape inner quotes
print(f'KPK deaths: {ps[ps["province"]=="Khyber Pakhtunkhwa"]["Deaths"].values[0]}')
print(f'Punjab deaths: {ps[ps["province"]=="Punjab"]["Deaths"].values[0]}')

## Gender Breakdown of Deaths & Injuries


Disaster research consistently shows that women and children face higher vulnerability during floods due to:
- Lower mobility (carrying children, less access to information)
- Cultural restrictions on movement during emergencies
- Higher risk of waterborne disease in camps

**Key finding to highlight:**
Children represent approximately 25% of total deaths — disproportionate to their share of the general population, indicating high vulnerability.


In [ ]:
fig2 = make_subplots(rows=1, cols=2,
    subplot_titles=["Deaths by Gender & Province", "Injuries by Gender & Province"],
    horizontal_spacing=0.12)

for name, cols, col_idx in [
    ("Deaths",  ["Deaths_Male","Deaths_Female","Deaths_Children"],  1),
    ("Injured", ["injured_male","injured_female","injured_children"], 2),
]:
    if col_idx == 2:
        ps2 = df.groupby("province").agg(
            injured_male=("injured_male","sum"),
            injured_female=("injured_female","sum"),
            injured_children=("injured_children","sum"),
        ).reset_index().sort_values("injured_male", ascending=False)
        src  = ps2
        ycols = ["injured_male","injured_female","injured_children"]
    else:
        src   = ps.sort_values("Deaths", ascending=False)
        ycols = ["Deaths_Male","Deaths_Female","Deaths_Children"]

    gcolors = ["#4FC3F7","#FF80AB","#FFD740"]
    glabels = ["Male","Female","Children"]
    for yc, gc, gl in zip(ycols, gcolors, glabels):
        fig2.add_trace(go.Bar(
            name=gl, x=src["province"], y=src[yc],
            marker_color=gc, opacity=0.88,
            showlegend=(col_idx==1),
            hovertemplate=f"%{{x}}<br>{gl}: %{{y:,}}<extra></extra>",
        ), row=1, col=col_idx)

fig2.update_layout(**base_layout("<b>Gender Breakdown — Deaths & Injuries</b>", h=480))
fig2.update_layout(barmode="stack")
fig2.update_xaxes(**AXIS, tickangle=-25)
fig2.update_yaxes(**AXIS)
fig2.show()

total_children = ps["Deaths_Children"].sum()
total_deaths   = ps["Deaths"].sum()
print(f"Children deaths: {total_children} ({total_children/total_deaths*100:.1f}% of total)")

## Top 10 Affected Districts (Bubble Chart)


**How to read this chart:**
- Position on Y axis = deaths (higher = more deadly)
- Bubble size = displaced (larger = more people forced to leave)
- Color = province

**The Buner outlier:**
Buner (256 deaths, July 13–28) is a single cloudburst event — one extreme rainfall event in one district in one day. This is a statistical outlier and should be noted in your methods section.

**Research insight:**
Large bubbles with low Y position = high displacement but fewer deaths (Punjab districts — riverine, slow).
Small bubbles with high Y position = many deaths but fewer displaced (KPK districts — flash, no time to escape).


In [ ]:
top10 = df.nlargest(10, "deaths").reset_index(drop=True)

fig3 = go.Figure(go.Scatter(
    x=top10["district"], y=top10["deaths"],
    mode="markers+text",
    marker=dict(
        size=top10["displaced"]/top10["displaced"].max()*80+15,
        color=[PROV_COLORS[p] for p in top10["province"]],
        opacity=0.85, line=dict(color="white", width=2),
    ),
    text=top10["deaths"], textposition="top center",
    textfont=dict(color=TEXT, size=12, family="Arial Black"),
    customdata=np.stack([top10["province"],top10["displaced"],
                         top10["flood_type"],top10["damage_index"]], axis=-1),
    hovertemplate=(
        "<b>%{x}</b><br>Province: %{customdata[0]}<br>"
        "Deaths: %{y:,}<br>Displaced: %{customdata[1]:,}<br>"
        "Type: %{customdata[2]}<br>Damage index: %{customdata[3]:.3f}<extra></extra>"),
))

fig3.add_annotation(x="Buner",
    y=float(top10[top10["district"]=="Buner"]["deaths"].values[0])+22,
    text="Worst district — 256 deaths",
    font=dict(color="#FF4B4B", size=10, family="Arial Black"),
    showarrow=True, arrowhead=2, arrowcolor="#FF4B4B", ax=60, ay=-35)

fig3.update_layout(**base_layout(
    "<b>Top 10 Most Affected Districts</b><br><sup>Bubble size = displaced persons | Color = province</sup>",
    h=520))
fig3.update_xaxes(**AXIS, tickangle=-20)
fig3.update_yaxes(**AXIS, title="Deaths")
fig3.show()

## Rescue Operations



**Why this matters scientifically:**
Rescue operations data lets you compare *hazard exposure* (deaths + displaced) against *response capacity* (people rescued). Provinces with high deaths but low rescue numbers indicate a response gap.

**Key finding:**
Punjab had significantly more rescue operations and people rescued than KPK — because Punjab's slower riverine floods allowed time to organize evacuations, while KPK's flash floods struck too fast for effective rescue.


In [ ]:
fig4 = make_subplots(rows=1, cols=2,
    subplot_titles=["Rescue Operations by Province","People Rescued (thousands)"],
    horizontal_spacing=0.12)

fig4.add_trace(go.Bar(
    x=ps["province"], y=ps["Rescue_Ops"],
    marker=dict(color=[PROV_COLORS[p] for p in ps["province"]], line=dict(color=BG,width=1)),
    text=ps["Rescue_Ops"], textposition="outside",
    textfont=dict(color=TEXT, size=10),
    hovertemplate="<b>%{x}</b><br>Operations: %{y:,}<extra></extra>",
    name="Operations",
), row=1, col=1)

fig4.add_trace(go.Bar(
    x=ps["province"], y=ps["People_Rescued"]/1000,
    marker=dict(color=[PROV_COLORS[p] for p in ps["province"]],
                opacity=0.75, line=dict(color=BG,width=1)),
    text=(ps["People_Rescued"]/1000).round(0).astype(int),
    textposition="outside", textfont=dict(color=TEXT, size=10),
    hovertemplate="<b>%{x}</b><br>Rescued: %{y:.0f}K<extra></extra>",
    name="Rescued (K)",
), row=1, col=2)

fig4.update_layout(**base_layout("<b>Rescue Operations — Pakistan 2025 Floods</b>", h=480))
fig4.update_xaxes(**AXIS, tickangle=-25)
fig4.update_yaxes(**AXIS)
fig4.show()

# Summary statistics
total_ops = df["rescue_operations"].sum()
total_rescued = df["people_rescued"].sum()
efficiency = total_rescued / total_ops if total_ops > 0 else 0

print(f'Total rescue operations : {total_ops:,}')
print(f'Total people rescued    : {total_rescued:,}')
print(f'Average rescued per op  : {efficiency:.0f} people')
print(f'Rescue coverage         : {(total_rescued/df["displaced"].sum()*100):.1f}% of displaced')

## Relief & Medical Camps


**What relief camps represent:**
Each camp is a temporary shelter providing food, clean water, medical care, and protection to displaced families. The number of camps is a proxy for *government response capacity and humanitarian reach*.

**Research question this answers:**
Is relief proportional to need? Compare deaths/displaced per province (Charts 1–2) against camps (this chart) to identify provinces that were under-served relative to their impact.


In [ ]:
fig5 = go.Figure(go.Bar(
    y=ps["province"], x=ps["Relief_Camps"],
    orientation="h",
    marker=dict(color=[PROV_COLORS[p] for p in ps["province"]],
                line=dict(color=BG, width=1)),
    text=ps["Relief_Camps"], textposition="outside",
    textfont=dict(color=TEXT, size=11),
    hovertemplate="<b>%{y}</b><br>Relief camps: %{x:,}<extra></extra>",
))
fig5.update_layout(**base_layout("<b>Relief & Medical Camps by Province</b>", h=430))
fig5.update_xaxes(**AXIS, title="Number of relief camps")
fig5.update_yaxes(**AXIS)
fig5.show()

# Summary statistics
total_camps = df["relief_camps"].sum()
total_displaced = df["displaced"].sum()
people_per_camp = total_displaced / total_camps if total_camps > 0 else 0

print(f'Total relief camps      : {total_camps:,}')
print(f'Average people per camp : {people_per_camp:.0f}')
print(f'Most camps (province)   : {ps.loc[ps["Relief_Camps"].idxmax(), "province"]} ({ps["Relief_Camps"].max():,} camps)')

## Deaths by Flood Type (Donut Chart)



**Flood type definitions:**
| Type | Mechanism | Speed | Deadliness |
|---|---|---|---|
| Flash Flood | Intense rain → instant runoff in mountains | Minutes | Very high |
| Flash Flood / Cloudburst | Extreme localized downpour (>100mm/hr) | Minutes | Extreme |
| Riverine Flood | River overflows flat plains | Days–weeks | Lower |
| GLOF | Glacial lake bursts | Hours | High |
| Flash Flood / Riverine | Mountain flood joins river | Hours | High |

**Key finding:**
Flash floods (all types combined) account for the majority of deaths despite affecting fewer districts — confirming that flood *type* is a stronger predictor of mortality than flood *extent*.


In [ ]:
ft = df.groupby("flood_type").agg(
    deaths=("deaths","sum"), districts=("district","count")
).reset_index().sort_values("deaths", ascending=False)

DCOL = ["#FF4B4B","#4FC3F7","#FFD740","#69F0AE","#CE93D8",
        "#80DEEA","#FF8A65","#A5D6A7","#90CAF9"]

fig6 = go.Figure(go.Pie(
    labels=ft["flood_type"], values=ft["deaths"],
    hole=0.58,
    marker=dict(colors=DCOL[:len(ft)], line=dict(color=BG, width=2)),
    textinfo="label+percent",
    textfont=dict(color=TEXT, size=9),
    hovertemplate="<b>%{label}</b><br>Deaths: %{value:,}<br>%{percent}<extra></extra>",
))
fig6.add_annotation(
    text=f"<b>{df['deaths'].sum():,}</b><br>total deaths",
    font=dict(color=TEXT, size=13, family="Arial Black"), showarrow=False)
fig6.update_layout(**base_layout("<b>Deaths by Flood Type</b>", h=500))
fig6.update_layout(paper_bgcolor=BG, font=dict(color=TEXT))
fig6.show()

## Infrastructure & Agricultural Damage

**What this chart shows:**
Two panels comparing physical damage across provinces:
- Left panel: Roads damaged (km), Bridges destroyed (×10 for scale), Schools damaged
- Right panel: Cropland lost (thousands of hectares), Livestock killed

**Why infrastructure damage matters beyond the numbers:**
Road destruction isolates communities — rescue teams cannot reach them, supplies cannot get in. This secondary effect multiplies the human cost of the initial flood.

**Key finding for Punjab:**
Punjab shows dominant cropland loss despite having fewer deaths than KPK. This reflects the agricultural nature of Punjab's flood plains — when rivers overflow, they inundate millions of hectares of wheat and rice crops.

> *"Agricultural losses were concentrated in Punjab (X,000 ha), where riverine flooding submerged productive farmland along the Chenab, Ravi, and Sutlej river corridors."*


In [ ]:
fig7 = make_subplots(rows=1, cols=2,
    subplot_titles=["Infrastructure Damage","Agricultural & Livelihood Loss"],
    horizontal_spacing=0.12)

# Infrastructure
for name, col, color in [
    ("Roads (km)",   ps["Roads_km"],        "#FF6B6B"),
    ("Bridges x10",  ps["Bridges"]*10,      "#4ECDC4"),
    ("Schools",      ps["Schools"],          "#45B7D1"),
]:
    fig7.add_trace(go.Bar(
        name=name, x=ps["province"], y=col,
        marker_color=color, opacity=0.88,
        hovertemplate=f"%{{x}}<br>{name}: %{{y:,.1f}}<extra></extra>",
    ), row=1, col=1)

# Agriculture
for name, col, color in [
    ("Cropland (K ha)", ps["Cropland_ha"]/1000, "#69F0AE"),
    ("Livestock (K)",   ps["Livestock"]/1000,   "#FFD740"),
]:
    fig7.add_trace(go.Bar(
        name=name, x=ps["province"], y=col,
        marker_color=color, opacity=0.88,
        hovertemplate=f"%{{x}}<br>{name}: %{{y:,.1f}}<extra></extra>",
    ), row=1, col=2)

fig7.update_layout(**base_layout("<b>Infrastructure & Agricultural Damage</b>", h=480))
fig7.update_layout(barmode="group")
fig7.update_xaxes(**AXIS, tickangle=-25)
fig7.update_yaxes(**AXIS)
fig7.show()

print(f"Total roads damaged  : {df['roads_km_damaged'].sum():.1f} km")
print(f"Total bridges lost   : {df['bridges_damaged'].sum():,}")
print(f"Total cropland (ha)  : {df['cropland_ha_damaged'].sum():,}")

##  Multi-Metric Heatmap (Normalized)

**What this chart shows:**
A normalized comparison matrix — every province compared on every metric simultaneously. The color scale runs Red (worst) to Green (best). Actual values are shown inside each cell.

**How normalization works:**
Each column is scaled 0–1 where 1.0 = the worst province for that metric.
This removes unit differences (you can't directly compare deaths vs km of roads) and shows *relative* severity.

**How to read it:**
- Read across a row → overall profile of one province
- Read down a column → which province was worst on that metric
- All red row → severely impacted across all dimensions (should be KPK or Punjab)


In [ ]:
metrics = ["Deaths","Injured","Displaced","Houses_Destroyed","Roads_km","Cropland_ha","Relief_Camps"]
mlabels = ["Deaths","Injured","Displaced","Houses","Roads","Cropland","Camps"]

hdf   = ps.set_index("province")[metrics]
hnorm = (hdf - hdf.min()) / (hdf.max() - hdf.min())

fig8 = go.Figure(go.Heatmap(
    z=hnorm.values, x=mlabels, y=hnorm.index.tolist(),
    colorscale="RdYlGn_r",
    text=hdf.values, texttemplate="%{text:,.0f}",
    textfont=dict(size=9, color="white"),
    hovertemplate="<b>%{y}</b><br>%{x}: %{text:,}<extra></extra>",
    showscale=True,
    colorbar=dict(
        title=dict(text="Normalized", font=dict(color=TEXT, size=10)),
        tickfont=dict(color=TEXT, size=9)),
))
fig8.update_layout(**base_layout(
    "<b>Multi-Metric Heatmap</b><br><sup>Red=worst | Actual values in cells | All metrics normalized 0-1</sup>",
    h=400))
fig8.update_xaxes(tickfont=dict(color=TEXT, size=11))
fig8.update_yaxes(tickfont=dict(color=TEXT, size=11))
fig8.update_layout(margin=dict(t=90,b=50,l=160,r=100))
fig8.show()

## Mortality Rate per 1,000 Displaced

**What this chart shows:**
A more nuanced lethality metric — deaths divided by displaced persons, multiplied by 1,000. This adjusts for population exposure and reveals which floods were most *deadly relative to their scale*.

**Why this is better than raw deaths for comparison:**
A district with 50 deaths but only 500 displaced (10% mortality rate) is experiencing a much more severe event than a district with 50 deaths and 50,000 displaced (0.1% mortality rate).

**Expected pattern:**
KPK flash flood districts should show high mortality rates (fast floods kill before people can escape). Punjab riverine districts should show low mortality rates (slow floods allow evacuation).

> *"Mortality rate was calculated as deaths per 1,000 displaced persons to normalize for population exposure, allowing cross-district comparison independent of district population size."*


In [ ]:
mr = df[df["mortality_rate"]>0].nlargest(15,"deaths").copy()

fig9 = go.Figure(go.Scatter(
    x=mr["district"], y=mr["mortality_rate"],
    mode="markers+lines",
    marker=dict(
        color=mr["mortality_rate"], colorscale="OrRd", size=13,
        line=dict(color="white", width=1.5), showscale=True,
        colorbar=dict(
            title=dict(text="Rate", font=dict(color=TEXT, size=10)),
            tickfont=dict(color=TEXT, size=9)),
    ),
    line=dict(color="#FF4B4B", width=1.5, dash="dot"),
    text=mr["mortality_rate"].round(1), textposition="top center",
    textfont=dict(color=TEXT, size=8),
    hovertemplate="<b>%{x}</b><br>Rate: %{y:.2f} per 1,000<extra></extra>",
))
fig9.update_layout(**base_layout(
    "<b>Mortality Rate per 1,000 Displaced</b><br><sup>High rate = deaths relative to displacement scale</sup>",
    h=480))
fig9.update_xaxes(**AXIS, tickangle=-30)
fig9.update_yaxes(**AXIS, title="Deaths per 1,000 displaced")
fig9.show()

##  Flood Event Timeline

**What this chart shows:**
The chronological progression of floods across Pakistan — when each district was hit, how many died, and how many were displaced (bubble size).

**The two phases clearly visible:**
- **Phase 1 (June 26 – July 31):** KPK-dominated, small bubbles (lower displacement) but higher Y positions (more deaths per district). Mean: 23 deaths/district.
- **Gap (July 27 – August 10):** Almost no events — the monsoon system physically shifted from NW mountains to eastern plains.
- **Phase 2 (August 1 – September 30):** Punjab/Sindh-dominated, large bubbles (massive displacement) but lower Y positions (fewer deaths per district). Mean: 10.5 deaths/district.

**This is a key scientific finding:**
The gap between phases proves these are two separate meteorological events, not one continuous flood — important for your methodology section.

> *"Temporal analysis revealed two distinct flood phases separated by an approximately 2-week interlude (July 27 – August 10), corresponding to the northwestward retreat and subsequent eastward shift of the monsoon system."*


In [ ]:
fig10 = go.Figure()

for prov in df["province"].unique():
    sub = df[df["province"]==prov].sort_values("start_date")
    fig10.add_trace(go.Scatter(
        x=sub["start_date"], y=sub["deaths"],
        mode="markers", name=prov,
        marker=dict(
            size=sub["displaced"]/df["displaced"].max()*55+7,
            color=PROV_COLORS[prov], opacity=0.85,
            line=dict(color="white", width=0.8)),
        customdata=sub[["district","displaced","flood_type"]].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>Date: %{x|%b %d}<br>"
            "Deaths: %{y:,}<br>Displaced: %{customdata[1]:,}<br>"
            "Type: %{customdata[2]}<extra></extra>"),
    ))

fig10.add_vrect(x0="2025-06-20", x1="2025-07-31",
    fillcolor="#FF4B4B", opacity=0.08,
    annotation_text="Phase 1 — KPK Flash Floods",
    annotation_position="top left",
    annotation_font=dict(color="#FF4B4B", size=10))
fig10.add_vrect(x0="2025-08-15", x1="2025-09-15",
    fillcolor="#4FC3F7", opacity=0.08,
    annotation_text="Phase 2 — Punjab Riverine",
    annotation_position="top left",
    annotation_font=dict(color="#4FC3F7", size=10))

fig10.update_layout(**base_layout(
    "<b>Flood Event Timeline</b><br><sup>Bubble size = displaced persons</sup>", h=500))
fig10.update_xaxes(**AXIS, title="Date")
fig10.update_yaxes(**AXIS, title="Deaths")
fig10.show()
print("Phase 1 (Jun-Jul): KPK flash floods — fast and deadly")
print("Phase 2 (Aug-Sep): Punjab riverine — large displacement")

##  Province Risk Radar


**How to read it:**
- Each axis goes from center (0 = least affected) to edge (1.0 = most affected)
- A province that fills more area = more severely affected overall
- The *shape* of each province's polygon reveals its impact pattern

**Expected shapes:**
- KPK → large on Deaths and Roads (flash floods kill and destroy mountain infrastructure)
- Punjab → large on Cropland and Displaced (riverine floods cover flat agricultural land)
- GB → moderate on Roads (GLOF floods destroy mountain roads disproportionately)

> *"Radar analysis revealed distinct provincial impact signatures — KPK exhibited a mortality-infrastructure profile characteristic of flash flooding, while Punjab showed a displacement-agricultural profile consistent with riverine inundation of flat cropland."*


In [ ]:
rm = ["Deaths","Displaced","Houses_Destroyed","Roads_km","Cropland_ha"]
rl = ["Deaths","Displaced","Houses","Roads","Cropland"]

rn = ps.copy()
for m in rm:
    rn[m] = (rn[m]-rn[m].min())/(rn[m].max()-rn[m].min()+1e-9)

fig11 = go.Figure()
for _, row in rn.iterrows():
    vals = [row[m] for m in rm]+[row[rm[0]]]
    fig11.add_trace(go.Scatterpolar(
        r=vals, theta=rl+[rl[0]], fill="toself",
        name=row["province"],
        line=dict(color=PROV_COLORS[row["province"]], width=2.5),
        fillcolor=PROV_COLORS[row["province"]], opacity=0.22,
        hovertemplate=f"<b>{row['province']}</b><br>%{{theta}}: %{{r:.2f}}<extra></extra>",
    ))

fig11.update_layout(
    **{k:v for k,v in base_layout("<b>Province Risk Profile Radar</b>",h=560).items()
       if k not in ("xaxis","yaxis","plot_bgcolor")},
    polar=dict(
        bgcolor=CARD,
        radialaxis=dict(gridcolor=GRID,linecolor=GRID,
                        tickfont=dict(color=MUTED,size=8),range=[0,1.1]),
        angularaxis=dict(gridcolor=GRID,linecolor=GRID,
                         tickfont=dict(color=TEXT,size=11))),
)
fig11.show()
print("KPK: dominant on Deaths and Roads")
print("Punjab: dominant on Cropland and Displaced")

##  Combined Publication Dashboard

**What this chart shows:**
1. **Top-left:** Deaths vs Displaced grouped bar — shows the inverse relationship between mortality and displacement by province
2. **Top-right:** Top 5 districts by Damage Index — the composite severity ranking
3. **Bottom-left:** Damage Index vs Deaths scatter with trend line — tests whether the composite index correlates with mortality
4. **Bottom-right:** Flood duration vs Deaths scatter — does longer flooding = more deaths?

**The trend line (bottom-left):**
A positive slope confirms the Damage Index is a valid predictor of deaths — validating your composite methodology.

> *"Figure X: Composite analysis dashboard showing (A) provincial human impact, (B) district damage ranking, (C) correlation between composite damage index and mortality (R²=X), and (D) relationship between flood duration and death toll."*

In [ ]:
fig12 = make_subplots(rows=2, cols=2,
    subplot_titles=["Deaths vs Displaced by Province",
                    "Top 5 Districts by Damage Index",
                    "Damage Index vs Deaths (trend)",
                    "Flood Duration vs Deaths"],
    vertical_spacing=0.18, horizontal_spacing=0.12)

# 1 — Deaths + displaced grouped bar
fig12.add_trace(go.Bar(
    name="Deaths", x=ps["province"], y=ps["Deaths"],
    marker_color=[PROV_COLORS[p] for p in ps["province"]], opacity=0.9,
    hovertemplate="%{x}<br>Deaths: %{y:,}<extra></extra>",
), row=1, col=1)
fig12.add_trace(go.Bar(
    name="Displaced (K)", x=ps["province"], y=ps["Displaced"]/1000,
    marker_color=[PROV_COLORS[p] for p in ps["province"]], opacity=0.4,
    hovertemplate="%{x}<br>Displaced: %{y:.0f}K<extra></extra>",
), row=1, col=1)

# 2 — Top 5 horizontal bar
top5 = df.nlargest(5,"damage_index")
fig12.add_trace(go.Bar(
    x=top5["damage_index"], y=top5["district"],
    orientation="h",
    marker=dict(color=top5["damage_index"], colorscale="RdYlGn_r",
                line=dict(color=BG, width=1)),
    text=top5["damage_index"].round(3), textposition="outside",
    textfont=dict(color=TEXT, size=10),
    hovertemplate="<b>%{y}</b><br>Index: %{x:.3f}<extra></extra>",
    name="Damage Index",
), row=1, col=2)

# 3 — Scatter damage_index vs deaths
fig12.add_trace(go.Scatter(
    x=df["damage_index"], y=df["deaths"], mode="markers",
    marker=dict(size=df["displaced"]/df["displaced"].max()*30+5,
                color=[PROV_COLORS[p] for p in df["province"]],
                opacity=0.75, line=dict(color="white", width=0.8)),
    text=df["district"],
    hovertemplate="<b>%{text}</b><br>Index: %{x:.3f}<br>Deaths: %{y:,}<extra></extra>",
    showlegend=False,
), row=2, col=1)
z = np.polyfit(df["damage_index"], df["deaths"], 1)
xl = np.linspace(df["damage_index"].min(), df["damage_index"].max(), 50)
fig12.add_trace(go.Scatter(
    x=xl, y=np.poly1d(z)(xl), mode="lines",
    line=dict(color="#FFD740", width=2, dash="dash"),
    name="Trend", showlegend=False,
), row=2, col=1)

# 4 — Duration vs deaths
fig12.add_trace(go.Scatter(
    x=df["flood_duration_days"], y=df["deaths"], mode="markers",
    marker=dict(size=df["houses_destroyed"]/df["houses_destroyed"].max()*25+5,
                color=[PROV_COLORS[p] for p in df["province"]],
                opacity=0.78, line=dict(color="white", width=0.8)),
    text=df["district"],
    hovertemplate="<b>%{text}</b><br>Duration: %{x}d<br>Deaths: %{y:,}<extra></extra>",
    showlegend=False,
), row=2, col=2)

fig12.update_layout(
    title=dict(text="<b>Pakistan 2025 Floods — Combined Analysis Dashboard</b>",
               font=dict(size=16, color=TEXT), x=0.5),
    paper_bgcolor=BG, plot_bgcolor=CARD, font=dict(color=TEXT),
    barmode="group", height=700,
    margin=dict(t=80,b=60,l=70,r=40),
    legend=dict(bgcolor=CARD, font=dict(color=TEXT, size=9)),
)
fig12.update_xaxes(gridcolor=GRID, tickfont=dict(color=MUTED, size=8), tickangle=-20)
fig12.update_yaxes(gridcolor=GRID, tickfont=dict(color=MUTED, size=8))
fig12.show()

##  Final Summary & Key Findings

In [ ]:
print("="*60)
print("  PAKISTAN 2025 MONSOON FLOOD — KEY FINDINGS")
print("="*60)
print(f"  Districts         : {len(df)}")
print(f"  Provinces         : {df['province'].nunique()}")
print(f"  Date range        : Jun 26 - Oct 1, 2025")
print(f"  Data sources      : NDMA + PDMA + OCHA + UNICEF + UNOSAT")
print()
print(f"  HUMAN IMPACT")
print(f"    Deaths          : {df['deaths'].sum():,}")
print(f"    - Male          : {df['deaths_male'].sum():,}")
print(f"    - Female        : {df['deaths_female'].sum():,}")
print(f"    - Children      : {df['deaths_children'].sum():,}")
print(f"    Injured         : {df['injured'].sum():,}")
print(f"    Displaced       : {df['displaced'].sum():,}")
print(f"    People rescued  : {df['people_rescued'].sum():,}")
print()
print(f"  INFRASTRUCTURE")
print(f"    Houses destroyed: {df['houses_destroyed'].sum():,}")
print(f"    Roads (km)      : {df['roads_km_damaged'].sum():.1f}")
print(f"    Bridges         : {df['bridges_damaged'].sum():,}")
print(f"    Schools         : {df['schools_damaged'].sum():,}")
print()
print(f"  AGRICULTURE")
print(f"    Cropland (ha)   : {df['cropland_ha_damaged'].sum():,}")
print(f"    Livestock       : {df['livestock_lost'].sum():,}")
print()
worst = df.nlargest(1,"deaths").iloc[0]
print(f"  WORST DISTRICT    : {worst['district']} ({worst['deaths']} deaths)")
print(f"  TOP DAMAGE INDEX  : {df.nlargest(1,'damage_index').iloc[0]['district']} ({df['damage_index'].max():.3f})")
print("="*60)
print("  All 12 charts generated | Dataset fully verified!")
print("="*60)

## Interactive District Map



**Why a geographic map matters for your paper:**
It shows *spatial clustering* — are deaths concentrated in specific geographic zones? The map should reveal that KPK deaths cluster along the Hindukush mountain valleys, while Punjab deaths spread along river corridors.

**Mapbox requirement:**
This chart uses Mapbox for the basemap. If it shows a blank map, change:
```python
mapbox_style="open-street-map"
```
No API key needed for OpenStreetMap.

> *"Figure X: Spatial distribution of flood-related mortality across 50 districts. Bubble size proportional to death toll. Geographic clustering in KPK reflects flash flood hazard zones along mountain river corridors."*


In [ ]:
fig13 = px.scatter_mapbox(
    df,
    lat='latitude', lon='longitude',
    size='deaths',
    color='province',
    color_discrete_map=PROV_COLORS,
    hover_name='district',
    hover_data={
        'deaths':True, 'injured':True,
        'houses_destroyed':True, 'roads_km_damaged':True,
        'relief_camps':True, 'latitude':False, 'longitude':False
    },
    size_max=55, zoom=4.2,
    center={'lat':30.3753,'lon':69.3451},
    mapbox_style='open-street-map',
    title='<b>Pakistan 2025 Flood Impact — District Map</b><br>'
          '<sup>Bubble size = deaths | Hover for details</sup>'
)
fig13.update_layout(**base_layout('', h=620))
fig13.update_layout(legend=dict(
    x=0.01, y=0.97,
    bgcolor='rgba(0,0,0,0.5)',
    font=dict(color='white', size=11)
))
fig13.show()
print(f'Districts mapped : {len(df)}')
print(f'Lat/lon range    : {df["latitude"].min():.2f} to {df["latitude"].max():.2f} N')


##  Child Vulnerability Analysis

**Why child vulnerability analysis matters:**
Children face higher flood mortality risk due to:
- Inability to swim or self-rescue
- Faster hypothermia onset
- Dependency on adults who may also be endangered
- Higher susceptibility to post-flood waterborne diseases

**SDG connection:**
This analysis connects to SDG 13 (Climate Action) and SDG 3 (Good Health) — disaster risk reduction must specifically protect children.

> *"Child mortality analysis revealed that children (under 18) accounted for X% of deaths in KPK, compared to X% in Punjab, suggesting differential age-vulnerability patterns between flash and riverine flood environments."*


In [ ]:
cv = df.groupby('province').agg(
    total_deaths     = ('deaths','sum'),
    child_deaths     = ('deaths_children','sum'),
    female_deaths    = ('deaths_female','sum'),
    male_deaths      = ('deaths_male','sum'),
).reset_index()
cv['child_pct']  = (cv['child_deaths']  / cv['total_deaths'] * 100).round(1)
cv['female_pct'] = (cv['female_deaths'] / cv['total_deaths'] * 100).round(1)
cv = cv.sort_values('child_pct', ascending=True)

fig14 = go.Figure()
fig14.add_trace(go.Bar(
    y=cv['province'], x=cv['male_deaths'],
    name='Male', orientation='h',
    marker_color='#4FC3F7', opacity=0.9,
    hovertemplate='%{y}<br>Male deaths: %{x}<extra></extra>'
))
fig14.add_trace(go.Bar(
    y=cv['province'], x=cv['female_deaths'],
    name='Female', orientation='h',
    marker_color='#F48FB1', opacity=0.9,
    hovertemplate='%{y}<br>Female deaths: %{x}<extra></extra>'
))
fig14.add_trace(go.Bar(
    y=cv['province'], x=cv['child_deaths'],
    name='Children', orientation='h',
    marker_color='#FFD54F', opacity=0.9,
    hovertemplate='%{y}<br>Child deaths: %{x}<extra></extra>'
))
fig14.update_layout(
    **base_layout('<b>Deaths by Gender & Age Group per Province</b>', h=420),
    barmode='stack'
)
fig14.update_xaxes(**AXIS, title_text='Deaths')
fig14.update_yaxes(**AXIS)
fig14.show()

print('Child deaths by province:')
for _, r in cv.sort_values('child_pct', ascending=False).iterrows():
    print(f'  {r["province"]:<25} {r["child_deaths"]:>4} children '
          f'({r["child_pct"]}% of {r["total_deaths"]} total)')


## Flood Duration vs Damage Correlation


**Hypothesis being tested:**
*H1: Longer flood duration is positively correlated with higher damage index.*

**Expected result:**
For riverine floods (Punjab, Sindh) — positive correlation expected. Rivers that stay high for weeks cause more structural damage than brief flash floods.
For flash floods (KPK) — short duration but possibly high damage index due to extreme intensity.

**Statistical note:**
If running this as a formal analysis, calculate Pearson's r or Spearman's rho between `flood_duration_days` and `damage_index`. Report the p-value.

> *"Pearson correlation analysis between flood duration and composite damage index yielded r=X (p=X), indicating [weak/moderate/strong] positive association between flood persistence and structural damage."*


In [ ]:
fd = df[df['flood_duration_days'] > 0].copy()

fig15 = px.scatter(
    fd,
    x='flood_duration_days', y='damage_index',
    size='deaths', color='province',
    color_discrete_map=PROV_COLORS,
    hover_name='district',
    hover_data={'deaths':True,'flood_duration_days':True,'damage_index':True},
    labels={
        'flood_duration_days':'Flood Duration (days)',
        'damage_index':'Damage Index (0-1)',
    },
    trendline='ols',
    trendline_scope='overall',
    trendline_color_override='#FF6B6B',
    title='<b>Flood Duration vs Damage Index</b><br>'
          '<sup>Bubble size = deaths | Red line = trend</sup>'
)
fig15.update_layout(**base_layout('', h=500))
fig15.update_traces(marker=dict(opacity=0.85))
fig15.show()

corr = fd['flood_duration_days'].corr(fd['damage_index'])
print(f'Correlation (duration vs damage index): {corr:.3f}')
print(f'Interpretation: {"Strong" if abs(corr)>0.6 else "Moderate" if abs(corr)>0.3 else "Weak"} '
      f'{"positive" if corr>0 else "negative"} relationship')


##  Houses Destroyed vs Partially Damaged


**Why the distinction matters:**
- Fully destroyed houses → families need complete reconstruction support (years)
- Partially damaged houses → families may be able to repair (months)

This has direct implications for recovery planning and budget requirements.

**Expected pattern:**
- KPK flash floods → higher proportion of full destruction (violent water + debris impact collapses structures)
- Punjab/Sindh riverine floods → higher proportion of partial damage (slow water inundation weakens but doesn't always collapse structures)
> *"Flash flood-affected districts in KPK showed a higher ratio of fully destroyed to partially damaged houses (X:1) compared to riverine flood districts in Punjab (X:1), reflecting the greater destructive force of flash floods on building structures."*


In [ ]:
hr = df.groupby('province').agg(
    full    = ('houses_destroyed','sum'),
    partial = ('houses_partially_damaged','sum')
).reset_index()
hr['total']       = hr['full'] + hr['partial']
hr['full_pct']    = (hr['full']    / hr['total'] * 100).round(1)
hr['partial_pct'] = (hr['partial'] / hr['total'] * 100).round(1)
hr = hr.sort_values('total', ascending=True)

fig16 = go.Figure()
fig16.add_trace(go.Bar(
    y=hr['province'], x=hr['full'],
    name='Fully Destroyed', orientation='h',
    marker_color='#FF4B4B', opacity=0.9,
    hovertemplate='%{y}<br>Fully destroyed: %{x:,}<extra></extra>'
))
fig16.add_trace(go.Bar(
    y=hr['province'], x=hr['partial'],
    name='Partially Damaged', orientation='h',
    marker_color='#FFA726', opacity=0.9,
    hovertemplate='%{y}<br>Partially damaged: %{x:,}<extra></extra>'
))
fig16.update_layout(
    **base_layout('<b>Houses Fully Destroyed vs Partially Damaged</b>', h=420),
    barmode='stack'
)
fig16.update_xaxes(**AXIS, title_text='Number of Houses')
fig16.update_yaxes(**AXIS)
fig16.show()

print('House damage ratio by province:')
for _, r in hr.sort_values('full_pct', ascending=False).iterrows():
    print(f'  {r["province"]:<25} Full={r["full_pct"]}%  '
          f'Partial={r["partial_pct"]}%  Total={r["total"]:,}')


In [ ]:
import geopandas as gpd
import pandas as pd
import plotly.graph_objects as go

# Load data
merged    = pd.read_csv("/home/saif/Desktop/flood/pakistan_flood_2025_WITH_FLOOD_EXTENT.csv")
districts = gpd.read_file("/home/saif/Desktop/flood/pak_admin_boundaries.shp/pak_admin2.shp")
districts = districts.to_crs(epsg=4326)

# Merge shapefile with flood data
districts["district_lower"] = districts["adm2_name"].str.strip().str.lower()
merged["district_lower"]    = merged["district"].str.strip().str.lower()

# Fix name mismatches
districts["district_lower"] = districts["district_lower"].replace({
    "tor ghar"          : "torghar",
    "kambar shahdad kot": "qambar shahdadkot",
    "kohistan upper"    : "kohistan",
})

geo = districts.merge(merged[["district_lower","flooded_km2","deaths","province"]],
                      on="district_lower", how="left")
geo["flooded_km2"] = geo["flooded_km2"].fillna(0)
geo["deaths"]      = geo["deaths"].fillna(0)

# Build choropleth map
fig = go.Figure()

# Base layer — all Pakistan districts (grey)
fig.add_trace(go.Choropleth(
    geojson     = geo.__geo_interface__,
    locations   = geo.index,
    z           = geo["flooded_km2"],
    colorscale  = "Blues",
    zmin=0, zmax=geo["flooded_km2"].max(),
    marker_line_color="#333", marker_line_width=0.5,
    colorbar=dict(
        title=dict(text="Flooded km²", font=dict(color="#E6EDF3")),
        tickfont=dict(color="#E6EDF3"),
        x=1.01,
    ),
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "Flooded area: %{z:.1f} km²<br>"
        "Deaths: %{customdata[1]:.0f}"
        "<extra></extra>"
    ),
    customdata=geo[["adm2_name","deaths"]].values,
))

# Bubble overlay — deaths per district
flood_only = merged[merged["flooded_km2"]>0].copy()
fig.add_trace(go.Scattergeo(
    lat=flood_only["latitude"],
    lon=flood_only["longitude"],
    mode="markers",
    marker=dict(
        size=flood_only["deaths"]/flood_only["deaths"].max()*50+5,
        color=flood_only["deaths"],
        colorscale="Reds",
        opacity=0.85,
        line=dict(color="white", width=0.8),
        showscale=False,
    ),
    text=flood_only["district"],
    customdata=flood_only[["deaths","flooded_km2","province"]].values,
    hovertemplate=(
        "<b>%{text}</b><br>"
        "Deaths: %{customdata[0]}<br>"
        "Flooded: %{customdata[1]:.1f} km²<br>"
        "Province: %{customdata[2]}"
        "<extra></extra>"
    ),
    name="Deaths (bubble size)",
))

fig.update_geos(
    fitbounds="locations",
    bgcolor="#0D1117",
    landcolor="#1a2332",
    oceancolor="#0D1117",
    showocean=True,
    showlakes=True, lakecolor="#0D1117",
    showrivers=True, rivercolor="#1E88E5",
    showcountries=True, countrycolor="#444",
    showcoastlines=True, coastlinecolor="#444",
)

fig.update_layout(
    title=dict(
        text="<b>Pakistan 2025 Monsoon Floods — Spatial Flood Mapping</b><br>"
             "<sup>Color = flooded area (km²) | Bubble = deaths | VIIRS 26 Aug–7 Sep 2025</sup>",
        font=dict(size=15, color="#E6EDF3"), x=0.5,
    ),
    paper_bgcolor="#0D1117",
    font=dict(color="#E6EDF3"),
    height=650,
    margin=dict(t=80, b=20, l=20, r=20),
    legend=dict(bgcolor="#161B22", font=dict(color="#E6EDF3")),
)

fig.show()
print("Map done!")

In [ ]:
import rasterio
import numpy as np
import geopandas as gpd
import pandas as pd
from rasterio.mask import mask

# Load the .tif file
tif_path = "/home/saif/Desktop/flood/Pakistan_Flood_Temporal_Zones_2025.tif"
districts = gpd.read_file("/home/saif/Desktop/flood/pak_admin_boundaries.shp/pak_admin2.shp")
districts = districts.to_crs("EPSG:4326")

results = []

with rasterio.open(tif_path) as src:
    print(f"CRS: {src.crs}")
    print(f"Shape: {src.shape}")
    
    for _, row in districts.iterrows():
        try:
            geom = [row.geometry.__geo_interface__]
            out_image, _ = mask(src, geom, crop=True, nodata=255)
            data = out_image[0]
            data = data[data != 255]
            
            total_pixels = len(data)
            if total_pixels == 0:
                continue
                
            phase1_pixels = np.sum(data == 1)  # KPK July
            phase2_pixels = np.sum(data == 2)  # Punjab August
            both_pixels   = np.sum(data == 3)  # Both phases
            
            # Convert pixels to km² (30m resolution)
            pixel_km2 = (30 * 30) / 1e6
            
            results.append({
                "district"      : row["adm2_name"],
                "phase1_km2"    : round(phase1_pixels * pixel_km2, 2),
                "phase2_km2"    : round(phase2_pixels * pixel_km2, 2),
                "both_km2"      : round(both_pixels   * pixel_km2, 2),
                "total_sar_km2" : round((phase1_pixels + phase2_pixels + both_pixels) * pixel_km2, 2),
            })
        except:
            continue

df_zones = pd.DataFrame(results)
df_zones = df_zones[df_zones["total_sar_km2"] > 0].sort_values("total_sar_km2", ascending=False)

print(f"\nDistricts with flood detected: {len(df_zones)}")
print(df_zones.head(15).to_string(index=False))

df_zones.to_csv("/home/saif/Desktop/flood/temporal_zones_per_district.csv", index=False)
print("\nSaved: temporal_zones_per_district.csv")

In [ ]:
import plotly.graph_objects as go

# Merge temporal zones with your main CSV
df_main  = pd.read_csv("/home/saif/Desktop/flood/pakistan_flood_2025_WITH_FLOOD_EXTENT.csv")
df_zones["district_lower"] = df_zones["district"].str.strip().str.lower()
df_main["district_lower"]  = df_main["district"].str.strip().str.lower()

# Fix name mismatches
df_zones["district_lower"] = df_zones["district_lower"].replace({
    "tor ghar"          : "torghar",
    "kambar shahdad kot": "qambar shahdadkot",
    "kohistan upper"    : "kohistan",
})

final = df_main.merge(
    df_zones[["district_lower","phase1_km2","phase2_km2","both_km2","total_sar_km2"]],
    on="district_lower", how="left"
).drop(columns=["district_lower"])

final[["phase1_km2","phase2_km2","both_km2","total_sar_km2"]] = \
    final[["phase1_km2","phase2_km2","both_km2","total_sar_km2"]].fillna(0)

final.to_csv("/home/saif/Desktop/flood/pakistan_flood_2025_FINAL_COMPLETE.csv", index=False)
print(f"Final dataset: {len(final)} districts, {len(final.columns)} columns")
print(f"Matched: {(final['total_sar_km2']>0).sum()} districts with SAR flood data")

# Chart — Phase 1 vs Phase 2 flooded area
matched = final[final["total_sar_km2"]>0].nlargest(15,"total_sar_km2")

fig = go.Figure()
fig.add_trace(go.Bar(name="Phase 1 — KPK Flash (July)",
    x=matched["district"], y=matched["phase1_km2"],
    marker_color="#FF4B4B", opacity=0.88))
fig.add_trace(go.Bar(name="Phase 2 — Punjab Riverine (Aug)",
    x=matched["district"], y=matched["phase2_km2"],
    marker_color="#4FC3F7", opacity=0.88))
fig.add_trace(go.Bar(name="Both phases",
    x=matched["district"], y=matched["both_km2"],
    marker_color="#CE93D8", opacity=0.88))

fig.update_layout(
    title=dict(
        text="<b>Temporal Flood Zonation — Phase 1 vs Phase 2</b><br>"
             "<sup>Red=KPK flash floods (July) | Blue=Punjab riverine (Aug-Sep) | Purple=both</sup>",
        font=dict(size=15,color="#E6EDF3"), x=0.5),
    paper_bgcolor="#0D1117", plot_bgcolor="#161B22",
    font=dict(color="#E6EDF3"), barmode="stack",
    xaxis=dict(gridcolor="#21262D", tickangle=-30, tickfont=dict(color="#8B949E")),
    yaxis=dict(gridcolor="#21262D", title="Flooded area (km²)", tickfont=dict(color="#8B949E")),
    legend=dict(bgcolor="#161B22", font=dict(color="#E6EDF3")),
    height=520,
)
fig.show()

print(f"\nKey temporal finding:")
print(f"  Phase 1 total (KPK July)    : {final['phase1_km2'].sum():,.0f} km²")
print(f"  Phase 2 total (Punjab Aug)  : {final['phase2_km2'].sum():,.0f} km²")
print(f"  Both phases                 : {final['both_km2'].sum():,.0f} km²")
print(f"  Total SAR flood extent      : {final['total_sar_km2'].sum():,.0f} km²")